# Lecture 12 — From scripts to an installable package

A package can be installed into an environment. Python then imports the installed package from any folder.

Notes: [Lecture 12](https://teaching.borisbolliet.io/c1-2026/materials/class12/notes.pdf) ·
code: [`attractor` at `lecture-12-start`](https://github.com/borisbolliet/attractor/tree/lecture-12-start/)

The notebook runs at the root of the `attractor` clone, at the tag `lecture-12-start`, with the kernel
*Python (attractor 3.13)*. `setup_env.sh` now also installs the project as an editable install:

```bash
cd ~/GitHub/attractor
git fetch --tags --force
git switch --detach lecture-12-start
source setup_env.sh
```

The cells that build wheels work in a temporary clone. The wheel cells never change the clone of the notebook.

In [ ]:
import importlib, importlib.util, json, math, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

TAG = "lecture-12-start"
REPO = "https://github.com/borisbolliet/attractor.git"
if not Path("src/attractor").exists():   # outside a clone (Colab, a download): fetch and install the project
    root = Path(tempfile.mkdtemp()) / "attractor"
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--branch", TAG, REPO, str(root)],
                   check=True)
    os.chdir(root)
    if importlib.util.find_spec("attractor") is None:   # e.g. Colab: install the project into this kernel
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[fit]"], check=True)
        sys.path.insert(0, str(root / "src"))
BIN = Path(sys.executable).parent        # the bin/ folder of the kernel's environment

def sh(cmd, cwd=None, tail=None):
    """Run a shell command, print its output (the last `tail` lines) and its exit status if not 0."""
    env = {**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PYTHON_COLORS": "0"}
    env["PATH"] = f"{BIN}{os.pathsep}{env['PATH']}"
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, env=env)
    lines = (r.stdout + r.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    if r.returncode:
        print(f"(exit status {r.returncode})")

subprocess.run(["git", "describe", "--tags"], capture_output=True, text=True).stdout.strip()

## 1 The import system

### The import search path

The **import search path** is a list of folders, `sys.path`. The statement `import name` uses `sys.path`
in three steps.

1. Python searches the folders of `sys.path` in order.
2. Python imports the first module or package called `name` that Python finds.
3. If no folder holds a module or package called `name`, Python raises `ModuleNotFoundError`.

For `python script.py`, the first entry is the folder of the script. For `python -c`, `python -m`
and the prompt, the first entry is the current folder. Then come the folders of `PYTHONPATH`. Then
comes the standard library. Last comes `site-packages`.

In [ ]:
for entry in sys.path:
    print(repr(entry))

In [ ]:
sh(f'{sys.executable} -c "import sys; print(repr(sys.path[0]))"')

For `python -c`, the first entry is `''`. The empty string stands for the current folder.

Python finds a file of the current folder before the standard library. In the next cell, a new
folder holds a file `random.py`:

In [ ]:
folder = Path(tempfile.mkdtemp())
(folder / "random.py").write_text('print("random.py of this folder")')
sh(f'{sys.executable} -c "import random"', cwd=folder)
sh(f'{sys.executable} -c "import random"')

In the new folder, the file `random.py` shadows the module `random` of the standard library. A
program that uses the standard module `random` then fails. Python raises an error for a missing
attribute:

In [ ]:
sh(f'{sys.executable} -c "import random; random.randint(1, 6)"', cwd=folder, tail=1)

In [ ]:
importlib.util.find_spec("no_such_module") is None

For `python script.py`, the folder of the script is first on `sys.path`. So the script finds only the
modules of its own folder:

In [ ]:
(folder / "scripts").mkdir()
(folder / "scripts" / "where.py").write_text("import sys; print(sys.path[0])")
sh(f"{sys.executable} scripts/where.py", cwd=folder)

In data analysis, a notebook may run in a directory containing a local copy of a module, such as `fit.py`. If that directory appears before `site-packages` in `sys.path`, Python imports the local copy instead of the installed version. The analysis then uses the local code, which may differ from the intended version, without raising an error.

### Packages and relative imports

A **package** is a module that contains other modules, its **submodules**. A regular package is a
folder that holds a file `__init__.py`. Importing the package runs the file `__init__.py`. The package
stores its folder in `__path__`. Python searches for the submodules of the package in the folder of `__path__`.

An **absolute import** names a module by its full name, as in `from pkg.mod import f`. A **relative
import** names a module from the package of the importing module, as in `from .mod import f`.

In [ ]:
hasattr(json, "__path__"), hasattr(math, "__path__")

In [ ]:
import json.decoder
json.decoder.__name__, json.decoder.__package__

In [ ]:
importlib.util.resolve_name(".decoder", "json"), importlib.util.resolve_name("..mod", "pkg.sub")

The modules of the project lie in [`src/attractor/`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/src/attractor/). The modules
import each other by relative imports, such as `from .trajectory import Trajectory` in
[`integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/src/attractor/integrators.py):

In [ ]:
print(Path("src/attractor/integrators.py").read_text().splitlines()[4])

A module of a package run as a file belongs to no package. Python raises `ImportError` at the first
relative import of the module:

In [ ]:
sh(f"{sys.executable} src/attractor/cli.py", tail=1)

### The src layout

In the **flat layout**, the package folder lies in the root of the project. In the **src layout**, the
package folder lies in `src/`. Commands run from the root of the project. The root is then the first
entry of `sys.path`. With the src layout, the root holds no package folder. As a result, `import
attractor` imports the installed package. In this notebook, the editable install makes Python find
the package in `src/`:

In [ ]:
import attractor
attractor.__file__

In [ ]:
sh(f'{sys.executable} -c "import attractor; print(attractor.__version__)"', cwd="/")

## 2 Project metadata and installation

### Distributions and the project file

A **distribution** is what pip installs. A distribution holds import packages and their
**metadata**. The metadata give the name, the version, the interpreters accepted and the distributions
required. The name of a distribution and
the name imported may differ.

In [ ]:
import importlib.metadata as md
md.packages_distributions()["dotenv"]

In [ ]:
meta = md.metadata("python-dotenv")
meta["Version"], meta["Requires-Python"], md.requires("python-dotenv")

A project declares its metadata in [`pyproject.toml`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/pyproject.toml). `[build-system]` names
the **build backend**. `[project]` holds the metadata. `[tool.name]` holds the settings of one tool.
In TOML, `[project.scripts]` is the table `scripts` inside the table `project`. The module `tomllib`
reads the file into nested dictionaries:

In [ ]:
import tomllib
doc = tomllib.loads(Path("pyproject.toml").read_text())
list(doc)

In [ ]:
doc["build-system"]

In [ ]:
doc["project"]["dependencies"], doc["project"]["optional-dependencies"], doc["project"]["scripts"]

In [ ]:
doc["tool"]

### Version specifiers and optional dependencies

Two versions are compared component by component, as integers. A missing component counts as 0. A
**version specifier** is a list of clauses separated by commas. A version satisfies the specifier
when the version satisfies all the clauses. `~=V` accepts V and the later versions that agree with V
in every component but the last.

In [ ]:
from packaging.specifiers import SpecifierSet
from packaging.version import Version

Version("1.10.1") > Version("1.9.3"), Version("1.15") == Version("1.15.0")

In [ ]:
releases = ["1.9.3", "1.10.1", "1.11.4", "1.15.3", "1.18.1"]
specs = ["~=1.11.2", "~=1.11", "==1.15", "==1.15.*", "!=1.11.*", ">=1.10,<1.15", "<=1.15"]
for spec in specs:
    print(f"{spec:<13}", list(SpecifierSet(spec).filter(releases)))

An **optional dependency** is a requirement of one feature only. The table
`[project.optional-dependencies]` groups optional dependencies under names, called **extras**. In the
metadata, an optional requirement names its extra:

In [ ]:
md.requires("attractor")

Only `attractor-fit` uses SciPy. Only the option `--plot` uses Matplotlib. So
[`cli.py`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/src/attractor/cli.py) imports each of the two libraries inside the
function that uses the library. As a result, `import attractor` loads neither SciPy nor Matplotlib:

In [ ]:
"scipy" in sys.modules, "matplotlib" in sys.modules

### Editable installs and lock files

An **editable install**, `pip install -e .`, works in three steps.

1. pip writes the metadata of the project into `site-packages`.
2. pip writes into `site-packages` a file ending in `.pth`, which names the folder `src/` of the project.
3. At each start-up of Python, the module `site` appends the folder `src/` to `sys.path`.

In [ ]:
import sysconfig
site_packages = Path(sysconfig.get_paths()["purelib"])
for pth in sorted(site_packages.glob("__editable__*attractor*.pth")):
    print(pth.name, "->", pth.read_text().strip())

In [ ]:
md.version("attractor"), attractor.__version__

`setup_env.sh` installs the lock first. `setup_env.sh` then installs the project without
dependencies. The command `pip check` then reports any unsatisfied requirement. Without `--no-deps`,
pip would replace a locked version that does not satisfy a requirement:

In [ ]:
sh(f"{sys.executable} -m pip check")

In [ ]:
sh(f'{sys.executable} -m pip install --dry-run "numpy<2.5"', tail=1)

A published result depends on the code, the data and the software environment that ran them. Reproducing the result later requires the same environment. A lock file records this environment.

## 3 Console commands and the public interface

### Console commands

An **entry point** is a named reference to an object of a package. An entry point is written
`module:attribute`. Each line of `[project.scripts]` defines a **console command**.

1. At install time, pip writes a **launcher** into the folder `bin/` of the environment.
2. When the command runs, the launcher imports the module and calls the function without arguments.
3. The launcher passes the return value to `sys.exit`. `None` gives the exit status 0.

In [ ]:
(ep,) = md.entry_points(group="console_scripts", name="dotenv")
ep.value, ep.module, ep.attr

In [ ]:
sorted(ep.name for ep in md.entry_points(group="console_scripts") if ep.module == "attractor.cli")

In [ ]:
print((BIN / "attractor-simulate").read_text())

The first line of the launcher names the interpreter of the environment by its absolute path. The package is installed in that environment. So the command runs from any folder. The next cell runs the
command from the root of the file system:

In [ ]:
sh("attractor-simulate", cwd="/")

Each function reads its options from `sys.argv`, or from a list. As a result, each function can
also be called from Python:

In [ ]:
from attractor.cli import simulate
simulate(["--h", "0.1"])

### The public interface

The **public interface** of a package is the set of names that other code may import from the
package and rely on. A package lists its public names in `__all__`. `__init__.py` imports the public
names from the submodules. Names that start with an underscore are private by convention.

In [ ]:
json.__all__, "codecs" in vars(json)

In [ ]:
names = {}
exec("from json import *", names)
sorted(name for name in names if name != "__builtins__")

The interface of `attractor`, in [`__init__.py`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/src/attractor/__init__.py):

In [ ]:
attractor.__all__

In [ ]:
attractor.RK4 is attractor.integrators.RK4, "rk4_step" in attractor.__all__

### Semantic versions

A **semantic version** `MAJOR.MINOR.PATCH` states what a release does to the public interface. A release
that breaks code written for the previous one increments MAJOR. A release that adds to the interface
and breaks nothing increments MINOR. A release that only fixes faults increments PATCH. NumPy 2.0.0
removed the alias `np.NaN`:

In [ ]:
import numpy as np
np.__version__

In [ ]:
np.NaN

In [ ]:
"1.26.4" in SpecifierSet("~=1.26"), "2.0.0" in SpecifierSet("~=1.26")

## 4 Wheels

### The contents of a wheel

A **wheel** is a zip archive that pip unpacks into `site-packages`. A wheel holds three kinds of files.

1. The import packages.
2. The declared package data.
3. A folder `name-version.dist-info`, with the files `METADATA`, `WHEEL`, `RECORD` and `entry_points.txt`.

A wheel does not hold tests, documentation or scripts outside the package.

In [ ]:
dist = md.distribution("python-dotenv")
print(dist.read_text("WHEEL").strip())
print(dist.read_text("entry_points.txt").strip())

The next cell builds the wheel of the project in a temporary clone at `lecture-12-start`. The
version is 1.0.0. pip first installs setuptools into a temporary environment. As a result, the cell
needs the network.

In [ ]:
work = Path(tempfile.mkdtemp())
sh(f"git clone -q {REPO} attractor && cd attractor && git -c advice.detachedHead=false switch -q --detach {TAG}",
   cwd=work)
clone = work / "attractor"
sh(f"{sys.executable} -m pip wheel . --no-deps -w dist -q", cwd=clone)
wheel100 = clone / "dist/attractor-1.0.0-py3-none-any.whl"
print("\n".join(zipfile.ZipFile(wheel100).namelist()))

In [ ]:
print("\n".join(zipfile.ZipFile(wheel100).read("attractor-1.0.0.dist-info/METADATA").decode().splitlines()[:12]))

### Package data

**Package data** are the files of a package that are not modules. setuptools puts a data file into the
wheel only if the data file is declared in `[tool.setuptools.package-data]`. The code of a package
reads a data file through `importlib.resources.files(package)`. The object returned by
`files(package)` represents the folder of the installed package:

In [ ]:
from importlib.resources import files
rc = files("matplotlib").joinpath("mpl-data/matplotlibrc")
rc.is_file(), rc.read_text().splitlines()[0]

In [ ]:
files("matplotlib").joinpath("mpl-data/no_such_file").is_file()

`lynx_hare` of [`datasets.py`](https://github.com/borisbolliet/attractor/blob/lecture-12-start/src/attractor/datasets.py) reads a table of the
package through `files("attractor")`. With the editable install, Python finds the file in `src/`:

In [ ]:
year, lynx, hare = attractor.lynx_hare()
len(year), int(year[0]), float(hare.max())

The wheel of 1.0.0 lacks the table. The next cell installs the wheel and its extra `fit` into a new
environment. In the new environment, `attractor-fit` fails on the missing file. pip downloads NumPy
and SciPy unless the cache of pip holds NumPy and SciPy.

In [ ]:
fresh = work / "fresh"
sh(f"{sys.executable} -m venv {fresh}")
sh(f'{fresh}/bin/python -m pip install -q "{wheel100}[fit]"')
sh(f"{fresh}/bin/attractor-fit", tail=1)

At `lecture-12`, two lines declare the table. The version becomes 1.0.1
([`pyproject.toml`](https://github.com/borisbolliet/attractor/blob/lecture-12/pyproject.toml)):

In [ ]:
sh("git -c advice.detachedHead=false switch -q --detach lecture-12 && git diff lecture-12-start lecture-12 -- pyproject.toml",
   cwd=clone)

In [ ]:
sh(f"{sys.executable} -m pip wheel . --no-deps -w dist -q", cwd=clone)
wheel101 = clone / "dist/attractor-1.0.1-py3-none-any.whl"
[name for name in zipfile.ZipFile(wheel101).namelist() if name.endswith(".csv")]

In [ ]:
sh(f'{fresh}/bin/python -m pip install -q "{wheel101}[fit]"')
sh(f"{fresh}/bin/attractor-fit")

The editable install of this notebook still reports 1.0.0. pip wrote the metadata of the editable
install at install time, at `lecture-12-start`.

### Compatibility tags

The last three fields of a wheel's name are its **compatibility tags**. The three tags are the Python
tag, the ABI tag and the platform tag. The interpreter accepts an ordered list of tags. pip installs a
wheel only if one of its tags is in the list. Among several compatible wheels, pip prefers the wheel
whose tag comes first.

In [ ]:
print(md.distribution("numpy").read_text("WHEEL"))

In [ ]:
from packaging.tags import sys_tags
accepted = [str(t) for t in sys_tags()]
len(accepted), accepted[:3]

The tag of a NumPy wheel compiled for CPython 3.12 is not in the list. The tag of the pure Python wheel
of the project is in the list:

In [ ]:
"cp312-cp312-macosx_14_0_arm64" in accepted, "py3-none-any" in accepted

Scientific software is useful to others only if it installs on their machines. The tags of a wheel decide which machines can install the package.

The field `requires-python` decides which interpreters may install the project. For example,
Python 3.10.18 does not satisfy the specifier of `requires-python`:

In [ ]:
spec = SpecifierSet(doc["project"]["requires-python"])
spec, "3.13.11" in spec, "3.10.18" in spec

The exercises and their solutions are at the end of the notes. The scripts of the exercises are in
[`exercises/lecture12/`](https://github.com/borisbolliet/attractor/tree/lecture-12/exercises/lecture12).